# AuraSight — YOLOv8 Acne Detection Training

一站式流程：下载数据 → 合并 + 重映射 → 检查标注 → 训练 → 评估 → 导出

**开始前：** Runtime → Change runtime type → **T4 GPU**

---

## Step 1: 安装依赖

In [ ]:
!pip install ultralytics roboflow -q

import os, shutil, yaml
from pathlib import Path
from collections import Counter
import torch
from ultralytics import YOLO
from roboflow import Roboflow

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2: 配置

只需要改 `ROBOFLOW_API_KEY`，其他都已经填好了。

想加更多数据集？往 `EXTRA_DATASETS` 列表里加就行，格式见注释。

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  只需要改这里                                  ║
# ╚══════════════════════════════════════════════╝

ROBOFLOW_API_KEY = "YOUR_API_KEY"  # ← Roboflow Settings → API Keys

# ── 你自己的数据集（主数据集，类名已经是 4 类）──
MY_DATASET = {
    "workspace": "aurasights-workspace",
    "project":   "acne-detection-v1-kf14t",
    "version":   2,  # ← 如果在 Roboflow 里重标后建了新 version，改这个数字
}

# ── 要合并的外部数据集（可以加任意多个）──
# 每个需要：workspace, project, version, class_map
# class_map: {源class名: 目标class名 或 None表示丢弃}
EXTRA_DATASETS = [
    # ── Dataset 1: acne-6rzah (8481 张, 12 类) ──
    {
        "workspace": "project-pcw5n",
        "project":   "acne-6rzah",
        "version":   1,
        "class_map": {
            "Blackhead":    "comedone",
            "Whitehead":    "comedone",
            "Milium":       "comedone",   # 外观像白头
            "Papular":      "papule",
            "Folliculitis": "pustule",    # 外观像脓疱
            "Purulent":     "pustule",
            "Conglobata":   "nodule",
            "Cystic":       "nodule",
            "Flat_wart":    None,          # 丢弃：不是痘痘
            "Keloid":       None,
            "Scars":        None,
            "Syringoma":    None,
        }
    },
    # ── Dataset 2: acne-new (3239 张, 6 类) ──
    {
        "workspace": "buyumedatasets",
        "project":   "acne-new",
        "version":   1,
        "class_map": {
            "acne":   "pustule",
            "pimple": "papule",
            "spot":   "comedone",
            "mole1":  None,    # 痣，不是痘痘
            "mole2":  None,
            "scar":   None,    # 疤痕
        }
    },
    # ── Dataset 3: Kritsakorn Acne (929 张, 6 类) ──
    {
        "workspace": "kritsakorn",
        "project":   "acne-kbm0q",
        "version":   5,
        "class_map": {
            "blackheads": "comedone",
            "whiteheads": "comedone",
            "papules":    "papule",
            "pustules":   "pustule",
            "nodules":    "nodule",
            "dark spot":  None,    # 色素沉着，不是活跃痘痘
        }
    },
    # ── 想加更多？复制上面的格式，填入新数据集的信息 ──
    # {
    #     "workspace": "xxx",
    #     "project":   "xxx",
    #     "version":   1,
    #     "class_map": {
    #         "原类名A": "comedone",
    #         "原类名B": "papule",
    #         "原类名C": None,  # 丢弃
    #     }
    # },
]

# 最终的 4 个类（不要改）
TARGET_NAMES = ["comedone", "papule", "pustule", "nodule"]
MERGED_DIR = Path("/content/merged_dataset")

print("✅ 配置完成")
print(f"   主数据集: {MY_DATASET['project']} v{MY_DATASET['version']}")
print(f"   外部数据集: {len(EXTRA_DATASETS)} 个")
for i, ds in enumerate(EXTRA_DATASETS):
    print(f"     {i+1}. {ds['project']} — {sum(1 for v in ds['class_map'].values() if v)} classes kept, {sum(1 for v in ds['class_map'].values() if not v)} dropped")
print(f"   目标类: {TARGET_NAMES}")

## Step 3: 下载所有数据集

In [ ]:
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# 下载主数据集
print("📥 Downloading main dataset...")
main_ds = rf.workspace(MY_DATASET["workspace"]).project(MY_DATASET["project"]).version(MY_DATASET["version"]).download("yolov8")
print(f"   ✅ {main_ds.location}")

# 下载外部数据集
extra_locations = []
for i, ds in enumerate(EXTRA_DATASETS):
    print(f"\n📥 Downloading extra dataset {i+1}: {ds['project']}...")
    loc = rf.workspace(ds["workspace"]).project(ds["project"]).version(ds["version"]).download("yolov8")
    extra_locations.append(loc.location)
    print(f"   ✅ {loc.location}")

print(f"\n✅ All datasets downloaded")

## Step 4: 合并 + 重映射

自动把所有数据集合并成一个，类名统一到 4 类。

In [ ]:
def find_image(img_dir, stem):
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        p = img_dir / (stem + ext)
        if p.exists(): return p
    return None

def copy_dataset(src_dir, out_dir, prefix, id_map=None):
    """
    复制一个数据集到 out_dir。
    id_map: {源class_id: 目标class_id}，None 的值会被丢弃。
           如果 id_map 为 None，表示不需要映射（class id 已经正确）。
    """
    src = Path(src_dir)
    total_img, kept_box, skip_box = 0, 0, 0

    for split in ["train", "valid", "test"]:
        img_in = src / split / "images"
        lbl_in = src / split / "labels"
        if not lbl_in.exists(): continue

        img_out = out_dir / split / "images"
        lbl_out = out_dir / split / "labels"
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)

        for lbl_file in sorted(lbl_in.glob("*.txt")):
            new_lines = []
            with open(lbl_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5: continue
                    old_id = int(parts[0])

                    if id_map is not None:
                        new_id = id_map.get(old_id)
                        if new_id is None:
                            skip_box += 1
                            continue
                        parts[0] = str(new_id)

                    new_lines.append(" ".join(parts))
                    kept_box += 1

            if not new_lines: continue

            stem = lbl_file.stem
            img_path = find_image(img_in, stem)
            if not img_path: continue

            new_name = prefix + stem
            shutil.copy2(img_path, img_out / (new_name + img_path.suffix))
            with open(lbl_out / (new_name + ".txt"), "w") as f:
                f.write("\n".join(new_lines) + "\n")
            total_img += 1

    return total_img, kept_box, skip_box

def build_id_map(src_dir, class_map):
    """
    从数据集的 data.yaml 读取源类名列表，
    结合 class_map 生成 {源id: 目标id} 映射。
    兼容 Roboflow 的两种 names 格式：list 和 dict。
    """
    yaml_path = Path(src_dir) / "data.yaml"
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    raw_names = cfg.get("names", [])

    # ── 兼容 dict 格式: {0: "Blackhead", 1: "Conglobata", ...} ──
    if isinstance(raw_names, dict):
        max_id = max(raw_names.keys())
        src_names = [raw_names.get(i, f"__unknown_{i}__") for i in range(max_id + 1)]
    else:
        src_names = list(raw_names)

    id_map = {}
    unmatched = []
    for i, name in enumerate(src_names):
        # 先精确匹配，再试大小写不敏感匹配
        target = class_map.get(name)
        if target is None:
            # 尝试 case-insensitive
            for k, v in class_map.items():
                if k.lower() == name.lower():
                    target = v
                    break
        if target is None and name not in class_map:
            # 这个类名在 class_map 里完全找不到（可能是拼写不同）
            unmatched.append(name)
            id_map[i] = None
        elif target is None:
            # 在 class_map 里但被显式设为 None（丢弃）
            id_map[i] = None
        else:
            id_map[i] = TARGET_NAMES.index(target)

    if unmatched:
        print(f"   ⚠️  WARNING: 这些类名在 class_map 里没找到，已丢弃: {unmatched}")
        print(f"      data.yaml 里的类名: {src_names}")
        print(f"      class_map 的 keys:  {list(class_map.keys())}")

    return id_map, src_names

# ── 开始合并 ──
if MERGED_DIR.exists():
    shutil.rmtree(MERGED_DIR)

print("="*55)
print("  Merging datasets")
print("="*55)

# 1. 主数据集 (不需要映射)
print(f"\n📦 Main: {MY_DATASET['project']} v{MY_DATASET['version']}")
imgs, kept, skipped = copy_dataset(main_ds.location, MERGED_DIR, "main_", id_map=None)
print(f"   → {imgs} images, {kept} boxes")
if imgs == 0:
    raise RuntimeError(f"❌ 主数据集 0 张图！检查路径: {main_ds.location}")

# 2. 外部数据集 (需要映射)
total_extra_imgs = 0
for i, (ds_cfg, ds_loc) in enumerate(zip(EXTRA_DATASETS, extra_locations)):
    print(f"\n📦 Extra {i+1}: {ds_cfg['project']}")
    id_map, src_names = build_id_map(ds_loc, ds_cfg["class_map"])

    mapped = {src_names[k]: TARGET_NAMES[v] for k, v in id_map.items() if v is not None}
    dropped = [src_names[k] for k, v in id_map.items() if v is None]
    print(f"   Mapped:  {mapped}")
    print(f"   Dropped: {dropped}")

    if not mapped:
        print(f"   ❌ ERROR: 没有任何类被映射！class_map 可能和 data.yaml 不匹配")
        print(f"      data.yaml names: {src_names}")
        print(f"      class_map keys:  {list(ds_cfg['class_map'].keys())}")
        raise RuntimeError(f"Dataset {ds_cfg['project']} 类名映射失败，请检查 class_map")

    imgs, kept, skipped = copy_dataset(ds_loc, MERGED_DIR, f"ext{i}_", id_map=id_map)
    print(f"   → {imgs} images, {kept} boxes kept, {skipped} dropped")

    if imgs == 0:
        print(f"   ⚠️  WARNING: 外部数据集 {ds_cfg['project']} 合并后 0 张图！")
    total_extra_imgs += imgs

if total_extra_imgs == 0:
    raise RuntimeError("❌ 所有外部数据集合并后都是 0 张图！请检查 class_map 配置。")

# 3. 写 data.yaml
with open(MERGED_DIR / "data.yaml", "w") as f:
    yaml.dump({
        "train": "../train/images",
        "val":   "../valid/images",
        "test":  "../test/images",
        "nc":    len(TARGET_NAMES),
        "names": TARGET_NAMES,
    }, f)

print(f"\n✅ Merged dataset: {MERGED_DIR}")
print(f"   主数据集: {imgs} images | 外部数据集: {total_extra_imgs} images")

## Step 5: 检查合并后的标注分布

确认类别比例合理再训练。

In [ ]:
import matplotlib.pyplot as plt

total_counts = Counter()
split_counts = {}

for split in ["train", "valid", "test"]:
    lbl_dir = MERGED_DIR / split / "labels"
    img_dir = MERGED_DIR / split / "images"
    counts = Counter()
    if lbl_dir.exists():
        for f in lbl_dir.glob("*.txt"):
            for line in open(f):
                parts = line.strip().split()
                if len(parts) >= 5:
                    counts[int(parts[0])] += 1
    n_img = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    split_counts[split] = (n_img, counts)
    for k, v in counts.items():
        total_counts[k] += v

total_boxes = sum(total_counts.values())

print("📊 Merged Dataset Summary")
print("=" * 55)
for split in ["train", "valid", "test"]:
    n_img, counts = split_counts[split]
    n_box = sum(counts.values())
    print(f"  {split:6s}: {n_img:5d} images, {n_box:6d} boxes")

print(f"\n  Class Distribution (all splits):")
print(f"  {'-'*50}")
for i, name in enumerate(TARGET_NAMES):
    c = total_counts.get(i, 0)
    pct = c / total_boxes * 100 if total_boxes > 0 else 0
    bar = '█' * int(pct / 2)
    print(f"    {name:12s}: {c:6d} ({pct:5.1f}%) {bar}")
print(f"    {'TOTAL':12s}: {total_boxes:6d}")

# 健康检查
nodule_pct = total_counts.get(3, 0) / total_boxes * 100 if total_boxes > 0 else 0
if nodule_pct > 25:
    print(f"\n  ⚠️  WARNING: nodule = {nodule_pct:.0f}%，临床上应该 <10%！")
    print(f"     标注可能有问题，训练前建议在 Roboflow 里修正。")
else:
    print(f"\n  ✅ 分布看起来合理")

# 图表
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#60a5fa', '#f59e0b', '#ef4444', '#8b5cf6']
vals = [total_counts.get(i, 0) for i in range(4)]
bars = ax.barh(TARGET_NAMES, vals, color=colors)
ax.set_xlabel('Count')
ax.set_title('Merged Dataset — Class Distribution')
for bar, c in zip(bars, vals):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2, str(c), va='center')
plt.tight_layout()
plt.show()

## Step 6: 训练 YOLOv8

用 `yolov8s`（小模型），平衡速度和精度。Early stopping 会在 20 个 epoch 无提升后自动停止。

In [ ]:
model = YOLO('yolov8s.pt')

results = model.train(
    data=str(MERGED_DIR / 'data.yaml'),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=15,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    mosaic=0.8,
    mixup=0.1,
    project='aurasight',
    name='acne_v2',
    exist_ok=True,
)

## Step 7: 评估 + 导出 + 保存（全自动，不会卡住）

In [ ]:
import glob

# ═══════════════════════════════════════════════
#  7a. 自动找到 best.pt
# ═══════════════════════════════════════════════
candidates = glob.glob('**/aurasight/acne_v2/weights/best.pt', recursive=True)
if not candidates:
    candidates = glob.glob('**/acne_v2/weights/best.pt', recursive=True)
if not candidates:
    candidates = glob.glob('**/weights/best.pt', recursive=True)
if not candidates:
    raise FileNotFoundError("找不到 best.pt！请检查 Step 6 是否训练成功。")

BEST_PT = candidates[0]
BEST_DIR = str(Path(BEST_PT).parent)
print(f"✅ Found model: {BEST_PT}")

best = YOLO(BEST_PT)

# ═══════════════════════════════════════════════
#  7b. 测试集评估
# ═══════════════════════════════════════════════
print("\n" + "="*55)
print("  Evaluating on test set...")
print("="*55)

metrics = best.val(
    data=str(MERGED_DIR / 'data.yaml'),
    split='test',
    imgsz=640,
)

print("\n📊 Test Results:")
print(f"  mAP@50:     {metrics.box.map50:.1%}")
print(f"  mAP@50-95:  {metrics.box.map:.1%}")
print(f"  Precision:  {metrics.box.mp:.1%}")
print(f"  Recall:     {metrics.box.mr:.1%}")

print("\nPer-class mAP@50:")
for i, name in enumerate(TARGET_NAMES):
    if i < len(metrics.box.ap50):
        print(f"  {name:12s}: {metrics.box.ap50[i]:.1%}")

# ═══════════════════════════════════════════════
#  7c. 可视化预测
# ═══════════════════════════════════════════════
import random
from PIL import Image

test_dir = MERGED_DIR / 'test' / 'images'
if test_dir.exists():
    test_images = [f for f in os.listdir(test_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    samples = random.sample(test_images, min(6, len(test_images)))

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, img_name in zip(axes.flat, samples):
        res = best.predict(str(test_dir / img_name), conf=0.3, imgsz=640, verbose=False)
        annotated = res[0].plot()
        ax.imshow(annotated[..., ::-1])
        ax.set_title(f"{img_name[:25]}... ({len(res[0].boxes)} det)")
        ax.axis('off')
    for ax in axes.flat[len(samples):]:
        ax.axis('off')
    plt.suptitle('YOLOv8 Predictions on Test Set', fontsize=16)
    plt.tight_layout()
    plt.show()

# ═══════════════════════════════════════════════
#  7d. 导出 ONNX + 本地保存
# ═══════════════════════════════════════════════
print("\n" + "="*55)
print("  Exporting to ONNX...")
print("="*55)

best.export(format='onnx', imgsz=640, simplify=True)

shutil.copy(BEST_PT, '/content/aurasight_acne_best.pt')
shutil.copy(os.path.join(BEST_DIR, 'best.onnx'), '/content/aurasight_acne_best.onnx')

print("✅ Models saved to /content/:")
print(f"  PyTorch: /content/aurasight_acne_best.pt")
print(f"  ONNX:    /content/aurasight_acne_best.onnx")

# 保存训练日志
results_csv = os.path.join(str(Path(BEST_PT).parent.parent), 'results.csv')
if os.path.exists(results_csv):
    shutil.copy(results_csv, '/content/aurasight_results.csv')
    print(f"  Log:     /content/aurasight_results.csv")

# ═══════════════════════════════════════════════
#  7e. Google Drive 保存（可选，失败不影响上面的结果）
# ═══════════════════════════════════════════════
print("\n" + "="*55)
print("  Saving to Google Drive...")
print("="*55)

try:
    from google.colab import drive
    drive.mount('/content/drive')

    save_dir = '/content/drive/MyDrive/AuraSight/models'
    os.makedirs(save_dir, exist_ok=True)

    shutil.copy('/content/aurasight_acne_best.pt', save_dir)
    shutil.copy('/content/aurasight_acne_best.onnx', save_dir)
    if os.path.exists('/content/aurasight_results.csv'):
        shutil.copy('/content/aurasight_results.csv', save_dir)

    print(f"✅ Saved to Google Drive: {save_dir}")
except Exception as e:
    print(f"⚠️ Google Drive 保存失败（不影响模型）: {e}")
    print(f"   模型仍在 /content/ 目录，手动下载即可。")

print("\n" + "="*55)
print("  🎉 ALL DONE!")
print("="*55)
print(f"\n  mAP@50 = {metrics.box.map50:.1%}")
print(f"  下载 /content/aurasight_acne_best.onnx")
print(f"  放到后端 AuraSight-api/models/ 目录")
print(f"  设置 YOLO_MODEL_PATH=./models/aurasight_acne_best.onnx")